# 04. Research Hypothesis 2: Topological & Spatial Relation Reasoning

**ARC Prize 2026 Research Project**  
This notebook evaluates whether explicit topological primitives (cavity enclosure, boundary tracing, region adjacency) and continuous raycasting primitives (directional line propagation, obstacle collision) improve held-out ARC generalization.

### Ablation Configurations (A through G):
- **A. Baseline v1**
- **B. Object Solver v1**
- **C. Baseline + Topology Only**
- **D. Baseline + Raycasting Only**
- **E. Baseline + Topology + Raycasting**
- **F. Object Solver + Topology**
- **G. Full Spatial-Object Solver**


In [ ]:
%matplotlib inline
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_task
from src.data.visualizer import plot_task
from src.topology.engine import enclosure_detection, flood_fill
from src.spatial.raycast import raycast

print("Spatial & Topology modules loaded successfully!")

## 1. Load Experimental Results

In [ ]:
results_file = project_root / "results" / "spatial_reasoning_v1.json"

with open(results_file, "r", encoding="utf-8") as f:
    results = json.load(f)

eval_split = results["eval_split"]
train_split = results["train_split"]
targeted_split = results["targeted_split"]

print(f"{'Solver Configuration':<38} | {'Eval Acc':<10} | {'Train Acc':<10} | {'Targeted Acc':<12}")
print("-" * 78)
for name in eval_res_keys := eval_split.keys():
    ev_acc = eval_split[name]['task_level_accuracy_pct']
    tr_acc = train_split[name]['task_level_accuracy_pct']
    tg_acc = targeted_split[name]['task_level_accuracy_pct']
    print(f"{name:<38} | {ev_acc:>6.2f}%    | {tr_acc:>6.2f}%    | {tg_acc:>8.2f}%")

## 2. Visualization of Spatial & Topological Primitives

In [ ]:
from src.data.models import Grid

# Enclosure & Cavity Fill Visualization
grid = Grid.from_list([
    [1, 1, 1, 0, 2, 2, 2],
    [1, 0, 1, 0, 2, 0, 2],
    [1, 1, 1, 0, 2, 2, 2],
])

enclosures = enclosure_detection(grid, background=0)
print(f"Detected Enclosed Cavities: {len(enclosures)}")
for i, enc in enumerate(enclosures):
    print(f"Cavity {i+1}: size={enc['region_size']}, boundary_colors={enc['boundary_colors']}")